# Shared enums - Python

All 8 Python examples from [docs/enums.md](https://platob.github.io/yggdryl/enums/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

In [ ]:
from yggdryl import DataType

value = DataType("int64")
assert value.id == "int64"
assert value.kind == "integer"

## Identity carries no parameters

In [ ]:
from yggdryl import DataType

stamp = DataType("timestamp(us, UTC)")
assert stamp.id == "timestamp"
assert str(stamp) == 'timestamp(us,"UTC")'

## MIME types

In [ ]:
from yggdryl import MimeType

parquet = MimeType.from_extension("parquet")
assert parquet == MimeType.PARQUET
assert str(parquet) == "application/vnd.apache.parquet"
assert parquet.top_level == "application"
assert parquet.is_tabular() and parquet.is_binary()

custom = MimeType("Application/Vnd.Example+JSON")
assert str(custom) == "application/vnd.example+json"
assert custom.structured_suffix == "json"
assert not custom.is_known()
assert custom.is_structured()

In [ ]:
import pytest

from yggdryl import MimeType

assert MimeType.from_content_type('Application/JSON; charset="utf-8"') == MimeType.JSON
with pytest.raises(ValueError):
    MimeType.from_content_type("application/json; charset")

assert MimeType.from_content_coding("x-gzip") == MimeType.GZIP
assert MimeType.GZIP.content_coding == "gzip"
with pytest.raises(ValueError):
    MimeType.from_content_coding("identity")

## Media types are a base plus its codings

In [ ]:
from yggdryl import MediaType, MimeType

media = MediaType.from_file_name("trades.json.gz")
assert media.base == MimeType.JSON
assert media.encodings == (MimeType.GZIP,)
assert media.encoding == MimeType.GZIP
assert media.extensions == ["json", "gz"]
assert media.is_encoded()
assert str(media) == "application/json;encodings=application/gzip"

In [ ]:
import pytest

from yggdryl import MediaType, MimeType

media = MediaType.from_content_headers("text/csv; charset=utf-8", "gzip")
assert media.base == MimeType.CSV
assert media.encoding == MimeType.GZIP

assert MediaType.from_extension("tgz") == MediaType.from_parts(MimeType.TAR, [MimeType.GZIP])

stacked = MediaType.from_file_name("events.json")
with pytest.raises(ValueError):
    stacked.push_encoding(MimeType.ZIP)
stacked.push_encoding(MimeType.ZSTD)
assert stacked.extensions == ["json", "zst"]

## Time units and union modes

In [ ]:
from yggdryl import DataType

assert str(DataType.time("microseconds")) == "time64(us)"
assert str(DataType.time("MICRO SECONDS")) == "time64(us)"
assert str(DataType.time("s")) == "time32(s)"

## Timezone

In [ ]:
import zoneinfo

from yggdryl import Timezone

# A name, an alias, and a zoneinfo all arrive at one value.
assert Timezone("Asia/Calcutta") == Timezone("Asia/Kolkata")
assert Timezone(zoneinfo.ZoneInfo("US/Eastern")) == Timezone("America/New_York")
assert Timezone("Z") == Timezone.UTC

sydney = Timezone("Australia/Sydney")
# Sydney's saving period spans the new year, so January is +11.
assert sydney.offset_at(1_705_000_000) == 11 * 3600
assert sydney.offset_at(1_720_000_000) == 10 * 3600
assert sydney.observes_saving()

# It duck-types as a tzinfo wherever only the offset is needed.
import datetime
assert sydney.utcoffset(1_720_000_000) == datetime.timedelta(hours=10)